# 3장. 데이터 누수와 자료 나누기

예측하는 시점에 알 수 없는 정보가 식에 들어가지 않았는가.

**준비:** 1장에서 쓴 전력수요·기온 CSV 다섯 개와 2장에서 쓴 서울 아파트 실거래가 CSV 네 개를 왼쪽 파일 창에 모두 올린 뒤 위에서부터 차례로 실행한다.

## 3.1 시계열 자료를 무작위로 나눌 때의 누수

In [ ]:
import glob, os, re
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)

def first_line(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        return f.readline()

csvs = sorted(glob.glob("*.csv"))
power_files = [f for f in csvs if first_line(f).startswith("날짜,1시")]
weather_files = [f for f in csvs if first_line(f).startswith("지점,지점명")]

wide = pd.concat([pd.read_csv(f, encoding="cp949") for f in power_files]).dropna(subset=["날짜"])
long = wide.melt("날짜", var_name="시", value_name="MWh")
long["시각"] = pd.to_datetime(long["날짜"]) + pd.to_timedelta(long["시"].str.replace("시", "").astype(int), unit="h")
power = long.set_index("시각")["MWh"].sort_index() / 1000          # GW

FIVE = ["서울", "대전", "대구", "부산", "광주"]
w = pd.concat([pd.read_csv(f, encoding="cp949", usecols=["지점명", "일시", "기온(°C)"]) for f in weather_files])
w["일시"] = pd.to_datetime(w["일시"])
temp = w[w["지점명"].isin(FIVE)].pivot(index="일시", columns="지점명", values="기온(°C)").interpolate()

HOLIDAYS = pd.to_datetime([
    "2023-01-01", "2023-01-21", "2023-01-22", "2023-01-23", "2023-01-24", "2023-03-01",
    "2023-05-05", "2023-05-27", "2023-05-29", "2023-06-06", "2023-08-15", "2023-09-28",
    "2023-09-29", "2023-09-30", "2023-10-02", "2023-10-03", "2023-10-09", "2023-12-25",
    "2025-01-01", "2025-01-27", "2025-01-28", "2025-01-29", "2025-01-30", "2025-03-01",
    "2025-03-03", "2025-05-05", "2025-05-06", "2025-06-03", "2025-06-06", "2025-08-15",
    "2025-10-03", "2025-10-05", "2025-10-06", "2025-10-07", "2025-10-08", "2025-10-09",
    "2025-12-25"])

h = pd.DataFrame({"GW": power, "기온": temp.mean(axis=1)}).dropna()
began = h.index - pd.Timedelta(hours=1)                           # 그 한 시간이 시작된 때
h["시"] = began.hour
h["쉬는날"] = ((began.dayofweek >= 5) | began.normalize().isin(HOLIDAYS)).astype(float)
h["시각번호"] = (h.index - pd.Timestamp("2023-01-01")) / pd.Timedelta(hours=1)
print(len(h))
print(h.iloc[[8, 9]].round(2).to_string())

In [ ]:
def rmse(actual, predicted):
    miss = np.asarray(actual, dtype=float) - np.asarray(predicted, dtype=float)
    return round(float(np.sqrt((miss ** 2).mean())), 2)

def by_clock(known, new, k=2):
    """시각번호가 가장 가까운 k시간을 known에서 찾아 그 수요를 평균한다. 다른 열은 쓰지 않는다."""
    T, Y = known["시각번호"].to_numpy(), known["GW"].to_numpy()
    return np.array([Y[np.argsort(np.abs(T - t))[:k]].mean() for t in new["시각번호"]])

def columns(df, extra=()):
    """시간대 가변수 24개, 쉬는 날의 시간대 가변수 24개, 19도보다 높은 정도, 15도보다 낮은 정도. extra의 열을 추가한다."""
    hour = (df["시"].to_numpy()[:, None] == np.arange(24)[None, :]).astype(float)
    parts = [hour, hour * df["쉬는날"].to_numpy()[:, None],
             np.maximum(df["기온"] - 19, 0), np.maximum(15 - df["기온"], 0)]
    return np.column_stack(parts + [df[c] for c in extra])

def by_line(known, new, extra=()):
    coef = np.linalg.lstsq(columns(known, extra), known["GW"].to_numpy(), rcond=None)[0]
    return columns(new, extra) @ coef

In [ ]:
held = np.random.default_rng(0).random(len(h)) < 0.2
mixed_train, mixed_test = h[~held], h[held]

print(len(mixed_train), len(mixed_test))
print("평균만", rmse(mixed_test["GW"], mixed_train["GW"].mean()))
print("선형식", rmse(mixed_test["GW"], by_line(mixed_train, mixed_test)))

In [ ]:
print("시계만", rmse(mixed_test["GW"], by_clock(mixed_train, mixed_test)))

In [ ]:
one = mixed_test.iloc[[1000]]
T = mixed_train["시각번호"].to_numpy()
near = mixed_train.iloc[np.argsort(np.abs(T - one["시각번호"].iloc[0]))[:2]].sort_index()
print(pd.concat([near.iloc[[0]], one, near.iloc[[1]]])[["GW", "기온", "시각번호"]].round(2).to_string())

In [ ]:
answer = by_clock(mixed_train, mixed_test)
T_train, T_test = mixed_train["시각번호"].to_numpy(), mixed_test["시각번호"].to_numpy()
both = np.isin(T_test - 1, T_train) & np.isin(T_test + 1, T_train)   # 앞뒤 한 시간이 모두 훈련 자료에 있는가

print(round(float(both.mean()), 3))
print("앞뒤가 모두 있을 때", rmse(mixed_test["GW"][both], answer[both]))
print("하나라도 없을 때", rmse(mixed_test["GW"][~both], answer[~both]))

In [ ]:
s = h.loc[h.index.year == 2023, "GW"].to_numpy()

def acf(lag):
    """lag시간 떨어진 두 시점의 수요 사이의 상관계수(자기상관)."""
    return float(np.corrcoef(s[lag:], s[:-lag])[0, 1])

print({lag: round(acf(lag), 3) for lag in [1, 2, 6, 12, 24, 168]})
print(round(float(s.std()), 2), round(float(s.std() * np.sqrt(1.5 - 2 * acf(1) + 0.5 * acf(2))), 2))

In [ ]:
train, test = h[h.index.year == 2023], h[h.index.year == 2025]

print(len(train), len(test))
print("평균만", rmse(test["GW"], train["GW"].mean()))
print("선형식", rmse(test["GW"], by_line(train, test)))
print("시계만", rmse(test["GW"], by_clock(train, test)))

In [ ]:
y23 = h[h.index.year == 2023]
began23 = (y23.index - pd.Timedelta(hours=1)).month                # 그 시간이 속한 달

def pooled(pairs, how):
    """(훈련, 시험) 쌍마다 제곱 오차를 구해 모두 모은 뒤 RMSE를 구한다."""
    sq = [(new["GW"].to_numpy() - how(known, new)) ** 2 for known, new in pairs]
    return round(float(np.sqrt(np.concatenate(sq).mean())), 2)

fold = np.random.default_rng(0).integers(0, 5, len(y23))
random_pairs = [(y23[fold != i], y23[fold == i]) for i in range(5)]
forward_pairs = [(y23[began23 < mth], y23[began23 == mth]) for mth in range(7, 13)]

rows = [(name, pooled(random_pairs, how), pooled(forward_pairs, how), rmse(test["GW"], how(train, test)))
        for name, how in [("선형식", by_line), ("시계만", by_clock)]]
print(pd.DataFrame(rows, columns=["방법", "무작위 5겹", "앞의 달로 다음 달", "2025년"]).set_index("방법").to_string())

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)

In [ ]:
# 그림 3-1
week = h.loc["2023-06-12 01:00":"2023-06-15 00:00"]
inside = held[h.index.get_indexer(week.index)]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(week.index, week["GW"], color=GRAY, lw=0.9, zorder=1)
ax.scatter(week.index[~inside], week["GW"][~inside], s=16, color=to_rgba(NAVY, 0.25), edgecolor=NAVY, lw=0.6, zorder=2, label="훈련 자료가 된 시간")
ax.scatter(week.index[inside], week["GW"][inside], s=30, color=BRICK, zorder=3, label="시험 자료가 된 시간")
ax.set_ylim(week["GW"].min() - 6, week["GW"].max() + 5)
ax.text(week.index[2], week["GW"].min() - 4.3, "벽돌색 점(시험)은 대부분 바로 앞뒤에 남색 점(훈련)이 있다.\n앞뒤 두 시간의 평균만으로도 그 시간의 수요를 거의 맞힌다", fontsize=8, color=BRICK)
ax.set_ylabel("전력수요(GW)", fontsize=8)
ax.set_xlabel("2023년의 시각(월-일 시)", fontsize=8)
ax.set_xticks(pd.date_range("2023-06-12", "2023-06-15", freq="12h"))
ax.set_xticklabels([t.strftime("%m-%d %H시") for t in pd.date_range("2023-06-12", "2023-06-15", freq="12h")], fontsize=7.5)
ax.legend(frameon=False, fontsize=8, loc="upper left", ncol=2)
frame(ax)
fig.tight_layout()
fig.savefig("fig_3_1.png", dpi=300)
plt.show()

In [ ]:
# 그림 3-2
lags = np.arange(1, 200)
vals = np.array([acf(int(lag)) for lag in lags])
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(lags, vals, width=0.9, color=to_rgba(NAVY, 0.25), edgecolor="none")
for lag, col in [(1, BRICK), (24, TEAL), (168, TEAL)]:
    ax.bar([lag], [acf(lag)], width=1.6, color=col)
ax.annotate(f"1시간 전: {acf(1):.2f}", (1, acf(1)), (14, 1.04), fontsize=8, color=BRICK, arrowprops={"arrowstyle": "->", "color": BRICK, "lw": 0.8})
ax.annotate(f"24시간 전: {acf(24):.2f}", (24, acf(24)), (38, 0.93), fontsize=8, color=TEAL, arrowprops={"arrowstyle": "->", "color": TEAL, "lw": 0.8})
ax.annotate(f"168시간(1주) 전: {acf(168):.2f}", (168, acf(168)), (128, 1.04), fontsize=8, color=TEAL, arrowprops={"arrowstyle": "->", "color": TEAL, "lw": 0.8})
ax.axhline(0, color=GRAY, lw=0.8)
ax.set_ylim(-0.25, 1.15)
ax.set_xticks(range(0, 193, 24))
ax.set_xlabel("두 시점 사이의 간격(시간)", fontsize=8)
ax.set_ylabel("자기상관계수", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_3_2.png", dpi=300)
plt.show()

## 3.2 예측 시점에 알 수 없는 특성

In [ ]:
def ago(hours):
    """hours시간 전의 수요. 그 시각의 기록이 없으면 빈 값."""
    return h["GW"].reindex(h.index - pd.Timedelta(hours=hours)).to_numpy()

g = h.copy()
g["한시간전"], g["어제"], g["그제"], g["지난주"] = ago(1), ago(24), ago(48), ago(168)
g["앞뒤평균"] = g["GW"].rolling(25, center=True, min_periods=1).mean()
g = g.dropna()
train, test = g[g.index.year == 2023], g[g.index.year == 2025]
print(len(train), len(test))

TRY = [(), ("한시간전",), ("앞뒤평균",), ("어제",), ("그제",), ("지난주",), ("그제", "지난주")]
rows = [(" + ".join(extra) or "없음", rmse(train["GW"], by_line(train, train, extra)),
         rmse(test["GW"], by_line(train, test, extra))) for extra in TRY]
hist = pd.DataFrame(rows, columns=["추가한 특성", "2023년", "2025년"]).set_index("추가한 특성")
print(hist.to_string())

In [ ]:
# 그림 3-5
order = ["한시간전", "앞뒤평균", "어제", "그제 + 지난주", "지난주", "그제", "없음"]
names = {"한시간전": "1시간 전의 수요", "앞뒤평균": "앞뒤 25시간의 평균", "어제": "24시간 전의 수요", "그제 + 지난주": "48시간 전 + 168시간 전",
         "지난주": "168시간 전의 수요", "그제": "48시간 전의 수요", "없음": "추가하지 않음(기본 식)"}
usable = {"한시간전": False, "앞뒤평균": False, "어제": False, "그제 + 지난주": True, "지난주": True, "그제": True, "없음": True}
fig, ax = plt.subplots(figsize=(7, 3.9))
ypos = np.arange(len(order))[::-1]
for yi, key in zip(ypos, order):
    val, ok = hist.loc[key, "2025년"], usable[key]
    col = TEAL if ok else BRICK
    ax.barh(yi, val, height=0.6, color=to_rgba(col, 0.45), edgecolor=col, lw=0.9)
    ax.text(val + 0.08, yi, f"{val:.2f}GW   " + ("전날 오전에 알 수 있다" if ok else "전날 오전에는 알 수 없다"), va="center", fontsize=8, color=col)
ax.set_yticks(ypos, [names[k] for k in order], fontsize=8)
ax.set_xlim(0, 7.6)
ax.set_xlabel("2025년의 RMSE(GW). 기본 식에 추가한 특성별", fontsize=8)
frame(ax, grid="x")
fig.tight_layout()
fig.savefig("fig_3_5.png", dpi=300)
plt.show()

## 3.3 전처리 순서에서 생기는 누수

In [ ]:
g["달평균_전체"] = g.groupby(g.index.to_period("M"))["GW"].transform("mean")
of_2023 = g[g.index.year == 2023].groupby(g[g.index.year == 2023].index.month)["GW"].mean()
g["달평균_훈련"] = g.index.month.map(of_2023).to_numpy()
train, test = g[g.index.year == 2023], g[g.index.year == 2025]

print(test.loc["2025-04-15 12:00", ["GW", "달평균_전체", "달평균_훈련"]].round(2).to_string())
for name in ["달평균_전체", "달평균_훈련"]:
    print(name, rmse(train["GW"], by_line(train, train, (name,))), rmse(test["GW"], by_line(train, test, (name,))))

## 3.4 같은 단지가 훈련 자료와 시험 자료에 모두 있을 때

In [ ]:
def period(path):
    with open(path, encoding="cp949", errors="ignore") as f:
        head = "".join(f.readline() for _ in range(15))
    return re.findall(r"계약일자 : (\d{4}-\d{2}-\d{2}) ~ (\d{4}-\d{2}-\d{2})", head)

deal_files = sorted(f for f in csvs if period(f))
raw = pd.concat([pd.read_csv(f, encoding="cp949", skiprows=15, thousands=",",
                             dtype={"번지": str, "본번": str, "부번": str, "동": str}) for f in deal_files],
                ignore_index=True)
v = raw[(raw["해제사유발생일"] == "-") & (raw["거래유형"] != "직거래")].copy()
v = v.rename(columns={"전용면적(㎡)": "면적"})
v["억"] = v["거래금액(만원)"] / 1e4
v["날짜"] = pd.to_datetime(v["계약년월"].astype(str) + v["계약일"].astype(str).str.zfill(2), format="%Y%m%d")
v["구"] = v["시군구"].str.split().str[1]
v["동네"] = v["시군구"].str.split().str[2]
v["단지"] = v["시군구"] + " " + v["번지"] + " " + v["단지명"]
v["집"] = v["단지"] + " " + v["면적"].round().astype(int).astype(str)
v["연식"] = v["날짜"].dt.year - v["건축년도"]
v = v.sort_values("날짜").reset_index(drop=True)

until_2024, in_2025 = v[v["날짜"].dt.year <= 2024], v[v["날짜"].dt.year == 2025]
print(len(deal_files), len(until_2024), len(in_2025), until_2024["단지"].nunique())

In [ ]:
GU, AGES = sorted(v["구"].unique()), [0, 10, 20, 30, 40, 50]

def off(actual, predicted):
    ratio = np.asarray(predicted, dtype=float) / np.asarray(actual, dtype=float) - 1
    return round(float(np.median(np.abs(ratio)) * 100), 1)

def onehot(s, cats):
    return (s.to_numpy()[:, None] == np.asarray(cats)[None, :]).astype(float)

def others(s):
    a = s.to_numpy()
    if len(a) == 1:
        return pd.Series([np.nan], index=s.index)
    return pd.Series([np.median(np.delete(a, i)) for i in range(len(a))], index=s.index)

def price(known, new, use_past):
    """known으로 식을 만들어 new의 값을 답한다. 전처리에 쓰는 값은 모두 known에서만 구한다."""
    known, new = known.copy(), new.copy()
    dongs = sorted(known["동네"].unique())
    of_house = known.groupby("집")["억"].median()
    of_dong, of_gu = known.groupby("동네")["억"].median(), known.groupby("구")["억"].median()
    known["지난가"] = known.groupby("집")["억"].transform(others)
    new["지난가"] = new["집"].map(of_house)

    def X(df):
        basic = [np.log(df["면적"]), df["층"], onehot(df["연식"].clip(0, 59) // 10 * 10, AGES)]
        if not use_past:
            return np.column_stack(basic + [onehot(df["동네"], dongs)[:, 1:]])
        filled = df["지난가"].fillna(df["동네"].map(of_dong)).fillna(df["구"].map(of_gu))
        return np.column_stack(basic + [onehot(df["구"], GU)[:, 1:], np.log(filled), df["지난가"].isna().astype(float)])

    coef = np.linalg.lstsq(X(known), np.log(known["억"].to_numpy()), rcond=None)[0]
    return np.exp(X(new) @ coef)

In [ ]:
rng = np.random.default_rng(0)

by_row = rng.random(len(until_2024)) < 0.2                          # 거래 단위로 20%를 무작위로 고른다
names = until_2024["단지"].unique()
picked = set(rng.choice(names, size=len(names) // 5, replace=False))  # 단지 단위로 20%를 무작위로 고른다
by_group = until_2024["단지"].isin(picked).to_numpy()

SPLITS = {
    "거래를 무작위로 나눔": (until_2024[~by_row], until_2024[by_row]),
    "단지 단위로 나눔": (until_2024[~by_group], until_2024[by_group]),
    "시간 순서로 나눔": (until_2024, in_2025),
}
rows = []
for name, (known, new) in SPLITS.items():
    seen = round(float(new["집"].isin(known["집"]).mean()), 3)
    rows.append((name, len(new), seen, off(new["억"], price(known, new, True)), off(new["억"], price(known, new, False))))
three = pd.DataFrame(rows, columns=["나누는 방법", "시험 건수", "훈련에 있는 집", "지난 가격을 쓰는 식", "동을 쓰는 식"]).set_index("나누는 방법")
print(three.to_string())

In [ ]:
# 그림 3-8
labels = {"거래를 무작위로 나눔": "거래를 무작위로 나눔\n(같은 집이 양쪽에 있다)", "단지 단위로 나눔": "단지 단위로 나눔\n(처음 보는 단지)", "시간 순서로 나눔": "시간 순서로 나눔\n(2025년의 거래)"}
fig, ax = plt.subplots(figsize=(7, 3.9))
ypos = np.arange(len(three))[::-1]
ax.barh(ypos + 0.19, three["지난 가격을 쓰는 식"], height=0.36, color=to_rgba(BRICK, 0.55), edgecolor=BRICK, lw=0.9, label="같은 집의 지난 가격을 쓰는 식")
ax.barh(ypos - 0.19, three["동을 쓰는 식"], height=0.36, color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=0.9, label="동을 가변수로 쓰는 식")
for yi, a, b, seen in zip(ypos, three["지난 가격을 쓰는 식"], three["동을 쓰는 식"], three["훈련에 있는 집"]):
    ax.text(a + 0.4, yi + 0.19, f"{a}%", va="center", fontsize=8, color=BRICK)
    ax.text(b + 0.4, yi - 0.19, f"{b}%", va="center", fontsize=8, color=NAVY)
    ax.text(38.5, yi, f"훈련 자료에 있는\n집의 비율 {seen * 100:.1f}%", va="center", ha="right", fontsize=7.5, color=TEAL)
ax.set_yticks(ypos, [labels[k] for k in three.index], fontsize=8)
ax.set_xlim(0, 39)
ax.set_ylim(-0.75, 3.05)
ax.set_xlabel("시험 자료에서 오차율의 중앙값(%)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper center", ncol=2, bbox_to_anchor=(0.45, 1.02))
frame(ax, grid="x")
fig.tight_layout()
fig.savefig("fig_3_8.png", dpi=300)
plt.show()